<a href="https://colab.research.google.com/github/fpellerano/devllm/blob/main/20_2_Prompt_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To upload a `requirements.txt` file to your Google Colab environment, follow these steps:

1.  **Open the Files pane**: Click the **Folder icon** in the left-hand sidebar.
2.  **Upload your file**:
    *   **Drag and drop** your file directly into the pane.
    *   **OR** click the **Upload to session storage** icon (a file with an upward arrow) to select it from your local machine.

  
>[IMPORTANT] Files uploaded this way are temporary and will be deleted once the session is recycled.

# Prompt Engineering with LangChain

**Prompt engineering** is the practice of designing inputs that steer a language model toward
the responses you want. Unlike fine-tuning, it requires no training — just careful wording.

This notebook covers the core techniques, built entirely on LangChain:
- The **system prompt** — the most important concept in applied LLM development
- **Zero-shot** and **few-shot** prompting
- **Chain-of-Thought** reasoning
- **Structured outputs** with Pydantic
- **Prompt sensitivity** — how wording affects outputs
- **Temperature** and its interaction with prompting strategies

In [ ]:
!pip install -qU -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1

In [ ]:
from google.colab import userdata

google_api_key = userdata.get("GOOGLE_API_KEY")

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    google_api_key=google_api_key,
    temperature=0.7,
)

## The System Prompt

The **system prompt** is a special message that runs before the conversation begins.
It is set by the developer — not the user — and the end user typically never sees it.

Think of it as the **policy** that governs every response the model will give:
- What persona or role should the model adopt?
- What format should responses follow?
- What topics are in or out of scope?
- What tone, language, or style is expected?

> **The system prompt is the single most powerful tool you have for shaping model behavior.**
> Swapping the system prompt on the same base model can turn a generic chatbot into a
> legal assistant, a customer support agent, a code reviewer, or a children's tutor —
> all without any fine-tuning.

When no system prompt is provided, the model falls back on its default behavior from training,
which is usually helpful but generic and often verbose. The examples below make this concrete.

### Baseline: No System Prompt vs. With a System Prompt

Let's ask the same question both ways and compare.

In [ ]:
question = "What is machine learning?"

# Without a system prompt — the model uses its default training behavior
print("=== No system prompt ===")
model.invoke([HumanMessage(question)]).pretty_print()

=== No system prompt ===
================================== Ai Message ==================================

[{'type': 'text', 'text': 'At its simplest, **machine learning (ML)** is a field of artificial intelligence (AI) that focuses on building systems that can learn from data, identify patterns, and make decisions with minimal human intervention.\n\nInstead of a human programmer writing specific rules (e.g., "If X happens, do Y"), machine learning involves feeding a computer large amounts of data and letting it figure out the rules for itself.\n\nHere is a breakdown of how it works and why it matters:\n\n---\n\n### 1. How it works (The Core Concept)\nTraditional programming is **explicit**: You provide the *data* and the *rules*, and the computer gives you the *answer*.\n\nMachine learning is **predictive**: You provide the *data* and the *answers* (the desired outcomes), and the computer calculates the *rules* (the algorithm) that connect them. Once those rules are learned, the compu

In [ ]:
# With a system prompt — we shape the response explicitly
print("=== With system prompt: concise expert ===")
model.invoke([
    SystemMessage("You are a machine learning expert. Answer in 2 sentences maximum. Be precise."),
    HumanMessage(question),
]).pretty_print()

=== With system prompt: concise expert ===
================================== Ai Message ==================================

[{'type': 'text', 'text': 'Machine learning is a field of artificial intelligence that focuses on building systems capable of learning patterns from data to make predictions or decisions without being explicitly programmed. It achieves this by using statistical algorithms to iteratively improve performance on a specific task as exposure to data increases.', 'extras': {'signature': 'EjQKMgG+Pvb7XimiLKT9ILoygJkgHD9jjW9jNfuS7YZFPf7uzoEFGjQEh7ZJYNmtB66IXgoJ'}}]


### Persona & Role Assignment

The system prompt can assign the model a **specific persona or role**. This shifts not just
the content of the response, but the vocabulary, tone, and framing — even for the exact same question.

In [ ]:
question = "What is machine learning?"

personas = {
    "PhD researcher": (
        "You are a research scientist specializing in machine learning. "
        "Use precise technical language. Assume the reader has a graduate-level background."
    ),
    "Teacher for beginners": (
        "You are a patient teacher explaining concepts to complete beginners. "
        "Use simple words, avoid jargon, and use an analogy."
    ),
    "Skeptical journalist": (
        "You are an investigative journalist writing about AI. "
        "Be curious but critical. Highlight both promise and hype."
    ),
}

for persona_name, system_content in personas.items():
    print(f"\n{'='*60}")
    print(f"PERSONA: {persona_name}")
    print('='*60)
    model.invoke([
        SystemMessage(system_content),
        HumanMessage(question),
    ]).pretty_print()


PERSONA: PhD researcher
================================== Ai Message ==================================

[{'type': 'text', 'text': 'Machine learning (ML) is a subfield of computer science and artificial intelligence focused on the development of algorithms that improve their performance on a specific task through empirical data rather than explicit, rule-based programming. Formally, it is the study of mathematical systems that learn a mapping function $f: \\mathcal{X} \\to \\mathcal{Y}$ by optimizing an objective function defined over a hypothesis space $\\mathcal{H}$.\n\nTo articulate this from a research perspective, we can decompose the discipline into several core components:\n\n### 1. The Learning Paradigm\nLearning is fundamentally an optimization problem. Given a dataset $\\mathcal{D} = \\{(\\mathbf{x}_i, y_i)\\}_{i=1}^n$, the objective is to find a parameter configuration $\\theta$ that minimizes a loss function $\\mathcal{L}(\\hat{y}, y)$, where $\\hat{y} = f_\\theta(\\mathb

### Format & Constraint Instructions

The system prompt is the right place to specify **exactly how** the model should structure
its output. This is critical for production systems where downstream code parses or renders responses.

Common patterns:
- `"Respond in exactly N bullet points."`
- `"Always reply in the language the user writes in."`
- `"Never mention competitors by name."`
- `"If you don't know the answer, say 'I don't know' — do not guess."`

In [ ]:
topic = "What are the benefits of exercise?"

formats = {
    "3 bullet points only": (
        "Respond with EXACTLY 3 bullet points. No introduction, no conclusion. "
        "Only the 3 bullets."
    ),
    "One sentence": (
        "Respond in exactly ONE sentence. No more, no less."
    ),
    "Pros and cons table (markdown)": (
        "Respond with a markdown table with two columns: 'Pro' and 'Con'. "
        "Include 3 rows. No text outside the table."
    ),
    "Respond only if confident": (
        "You may only answer questions you are highly confident about. "
        "If uncertain about any part, reply: 'I am not confident enough to answer this.' "
        "Do not guess or speculate."
    ),
}

for format_name, system_content in formats.items():
    print(f"\n{'='*60}")
    print(f"FORMAT: {format_name}")
    print('='*60)
    model.invoke([
        SystemMessage(system_content),
        HumanMessage(topic),
    ]).pretty_print()


FORMAT: 3 bullet points only
================================== Ai Message ==================================

[{'type': 'text', 'text': '* It improves cardiovascular health by strengthening the heart and enhancing circulation throughout the body.\n* It promotes mental well-being by reducing symptoms of anxiety and depression through the release of endorphins.\n* It helps maintain a healthy weight and increases physical strength, bone density, and metabolic function.', 'extras': {'signature': 'EjQKMgG+Pvb77RAzc8jcal4LHZkiNsfr75zzEHP+n88TA1q6uDXq9z7uWqb2MqZ78MFgW6Jp'}}]

FORMAT: One sentence
================================== Ai Message ==================================

[{'type': 'text', 'text': 'Regular exercise significantly improves physical health by strengthening the heart and muscles while simultaneously boosting mental well-being through the reduction of stress and anxiety.', 'extras': {'signature': 'EjQKMgG+Pvb7DlzxSMu5UkBgZ18K4xv68JlZr6mjZUmDqYvnQcdim/X5zRaDj3O3ALH2iBbj'}}]


### The System Prompt as an Application

The most important insight: **the system prompt is the product**.
The same underlying model becomes a completely different application depending on what
you put in the system prompt. Below we define two distinct "apps" built on the same model.

In [ ]:
# App 1: Customer support agent for a software company
customer_support_system = (
    "You are a friendly and empathetic customer support agent for CloudSync, "
    "a cloud storage company. Your goals are to: (1) understand the customer's issue, "
    "(2) provide a clear solution or workaround, (3) apologize sincerely for any inconvenience. "
    "Never mention competitors. If you cannot resolve an issue, offer to escalate to a human agent. "
    "Keep responses under 100 words."
)

# App 2: Senior code reviewer
code_reviewer_system = (
    "You are a senior software engineer conducting a code review. "
    "Be direct, technical, and constructive. For each issue found, specify: "
    "SEVERITY (low/medium/high), a brief explanation, and a concrete fix. "
    "If the code looks good, say so explicitly."
)

user_message = "I uploaded a file but it seems to have disappeared. What happened?"

print("=== App 1: Customer Support Agent ===")
model.invoke([
    SystemMessage(customer_support_system),
    HumanMessage(user_message),
]).pretty_print()

code_snippet = "def get_user(id):\n    user = db.query('SELECT * FROM users WHERE id=' + id)\n    return user"

print("\n=== App 2: Code Reviewer ===")
model.invoke([
    SystemMessage(code_reviewer_system),
    HumanMessage(f"Please review this Python function:\n\n{code_snippet}"),
]).pretty_print()

=== App 1: Customer Support Agent ===
================================== Ai Message ==================================

[{'type': 'text', 'text': 'I’m so sorry to hear your file went missing! I know how frustrating that is. \n\nFirst, please check your "Trash" or "Deleted Items" folder in the CloudSync dashboard to see if it was moved there by mistake. You can also try refreshing the page or logging out and back in to sync your view.\n\nIf the file is still not appearing, please let me know your account email address. I’d be happy to escalate this to our technical team so they can investigate the server logs for you!', 'extras': {'signature': 'EjQKMgG+Pvb7MsYLm6iGsL7WqvTjbD7oGmPuiFXr2cofKyJtRo9k9p3PH4cqiYHO+k3bTXJi'}}]

=== App 2: Code Reviewer ===
================================== Ai Message ==================================

[{'type': 'text', 'text': 'This code is **critically flawed** and must not be deployed. Below is the code review.\n\n### 1. SQL Injection Vulnerability\n*   **

## Zero-Shot Prompting

**Zero-shot prompting** means asking the model to perform a task without providing any examples.
You rely entirely on knowledge the model acquired during pre-training.

Modern LLMs handle a wide range of tasks zero-shot — classification, translation, summarization,
question answering — because they've seen so many examples of these tasks in training data.

Zero-shot is the right starting point: if it works, there's no need to add examples.
Add examples (few-shot) only when zero-shot is insufficient.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Zero-shot sentiment classification — no examples, just instructions
zero_shot_template = ChatPromptTemplate.from_messages([
    ("system",
     "You are a sentiment classifier. "
     "Classify the sentiment of the text as Positive, Negative, or Neutral. "
     "Reply with ONLY the label — no explanation."),
    ("user", "{text}"),
])

zero_shot_chain = zero_shot_template | model | StrOutputParser()

texts = [
    "I absolutely love this product, it exceeded all my expectations!",
    "The delivery was two weeks late and the item arrived damaged.",
    "The package was delivered on Tuesday.",
    "The battery life is decent but the camera is disappointing.",
]

print(f"{'Text':<55} | Label")
print("-" * 70)
for text in texts:
    label = zero_shot_chain.invoke({"text": text})
    print(f"{text[:53]:<55} | {label}")

Text                                                    | Label
----------------------------------------------------------------------
I absolutely love this product, it exceeded all my ex   | Positive
The delivery was two weeks late and the item arrived    | Negative
The package was delivered on Tuesday.                   | Neutral
The battery life is decent but the camera is disappoi   | Negative


## Few-Shot Prompting

**Few-shot prompting** provides the model with a small number of labeled examples directly
in the prompt, before the actual input. This technique:

- Guides the model to follow a specific output format or style
- Improves performance on tasks with ambiguous or non-standard labels
- Demonstrates the *pattern* you want, rather than just describing it

The examples act as an in-context specification — no gradient updates, no training.

### Few-Shot with `ChatPromptTemplate`

LangChain's `ChatPromptTemplate` lets you include example `(human, ai)` message pairs
as part of the prompt structure, making few-shot prompts clean and reusable.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Examples baked into the prompt as prior turns
few_shot_template = ChatPromptTemplate.from_messages([
    ("system",
     "You are a sentiment classifier. "
     "Classify the sentiment as Positive, Negative, or Neutral. "
     "Reply with ONLY the label."),
    # Few-shot examples as alternating user/assistant turns
    ("human",  "The product arrived on time and works perfectly."),
    ("ai",     "Positive"),
    ("human",  "Terrible quality, broke after one day."),
    ("ai",     "Negative"),
    ("human",  "The package was delivered on Thursday."),
    ("ai",     "Neutral"),
    ("human",  "Great value for money but the instructions were confusing."),
    ("ai",     "Neutral"),
    # The actual input
    ("human",  "{text}"),
])

few_shot_chain = few_shot_template | model | StrOutputParser()

# Test with the same inputs as before — compare to zero-shot
test_cases = [
    "I absolutely love this product, it exceeded all my expectations!",
    "The delivery was two weeks late and the item arrived damaged.",
    "The package was delivered on Tuesday.",
    "The battery life is decent but the camera is disappointing.",
]

print(f"{'Text':<55} | Zero-shot | Few-shot")
print("-" * 80)
for text in test_cases:
    zero = zero_shot_chain.invoke({"text": text})
    few  = few_shot_chain.invoke({"text": text})
    print(f"{text[:53]:<55} | {zero:<9} | {few}")

Text                                                    | Zero-shot | Few-shot
--------------------------------------------------------------------------------
I absolutely love this product, it exceeded all my ex   | Positive  | Positive
The delivery was two weeks late and the item arrived    | Negative  | Negative
The package was delivered on Tuesday.                   | Neutral   | Neutral
The battery life is decent but the camera is disappoi   | Negative  | Neutral


### Choosing Few-Shot Examples

The examples you choose matter. Poor examples can mislead the model.
Good examples should:
- **Cover edge cases** — ambiguous or mixed-sentiment text
- **Be representative** — reflect the distribution of real inputs
- **Be unambiguous** — if you're unsure of the label, don't include it
- **Be minimal** — 3–5 good examples typically outperform 10 mediocre ones

## Chain-of-Thought (CoT) Prompting

**Chain-of-Thought prompting** asks the model to *reason step by step* before producing
a final answer. Introduced by Wei et al. (2022), it significantly improves performance
on tasks that require multi-step reasoning: math, logic, commonsense inference.

The key insight: language models are not just retrieving facts — they're generating text
token by token. If you force them to generate intermediate reasoning steps, those steps
become part of the context that informs the final answer.

There are two variants:

| Variant | How | When to use |
|---|---|---|
| **Manual CoT** | Include reasoning in few-shot examples | Complex tasks, full control |
| **Zero-shot CoT** | Append *"Let's think step by step."* | Quick wins with no examples |

In [ ]:
# Manual CoT: examples include explicit reasoning chains
cot_template = ChatPromptTemplate.from_messages([
    ("system",
     "Classify each review as Positive, Negative, or Neutral. "
     "First, identify the key sentiment signals in the text, reason about the overall tone, "
     "then give the classification on a new line as: Classification: <label>"),
    ("human",  "The food was absolutely delicious and the service was outstanding."),
    ("ai",
     "Reasoning: 'absolutely delicious' and 'outstanding' are strongly positive words "
     "describing both food and service. No negative signals.\n"
     "Classification: Positive"),
    ("human",  "The delivery was late and the package was damaged."),
    ("ai",
     "Reasoning: 'late' and 'damaged' are clearly negative. "
     "Both aspects of the experience were bad.\n"
     "Classification: Negative"),
    ("human",  "The meeting was held on Tuesday as scheduled."),
    ("ai",
     "Reasoning: This is a factual statement. No evaluative or sentiment language present.\n"
     "Classification: Neutral"),
    ("human",  "{text}"),
])

cot_chain = cot_template | model | StrOutputParser()

# Ambiguous case where reasoning helps
test = "The hotel room was clean but the neighborhood was noisy at night."
print(f"Input: {test}\n")
print(cot_chain.invoke({"text": test}))

Input: The hotel room was clean but the neighborhood was noisy at night.

Reasoning: The review contains a positive sentiment ("clean") and a negative sentiment ("noisy at night"). Since the positive and negative aspects balance each other out, the overall tone is mixed or indifferent.

Classification: Neutral


In [ ]:
# Zero-shot CoT: just append "Let's think step by step."
# No examples needed — the phrase alone activates step-by-step reasoning.
zero_shot_cot_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful reasoning assistant."),
    ("user",   "{problem} Let's think step by step."),
])

zero_shot_cot_chain = zero_shot_cot_template | model | StrOutputParser()

problems = [
    "A train leaves city A at 9am traveling at 60 mph. Another leaves city B at 10am "
    "traveling toward city A at 80 mph. The cities are 210 miles apart. When do they meet?",
    "If all Bloops are Razzles, and all Razzles are Lazzles, are all Bloops definitely Lazzles?",
]

for problem in problems:
    print("Problem:", problem)
    print("\nAnswer:")
    print(zero_shot_cot_chain.invoke({"problem": problem}))
    print("\n" + "="*70 + "\n")

Problem: A train leaves city A at 9am traveling at 60 mph. Another leaves city B at 10am traveling toward city A at 80 mph. The cities are 210 miles apart. When do they meet?

Answer:
To find out when the two trains meet, we can break the problem down into steps:

### Step 1: Account for the head start
*   The first train (Train A) leaves at 9:00 am traveling at 60 mph.
*   The second train (Train B) leaves at 10:00 am.
*   By the time Train B starts moving, Train A has been traveling for 1 hour.
*   **Distance covered by Train A in that hour:** $60 \text{ mph} \times 1 \text{ hour} = 60 \text{ miles}$.

### Step 2: Determine the remaining distance
*   The total distance between City A and City B is 210 miles.
*   After the first hour, Train A has covered 60 miles, so the remaining distance between the two trains at 10:00 am is:
    $210 - 60 = 150 \text{ miles}$.

### Step 3: Calculate the closing speed
*   Starting at 10:00 am, both trains are moving toward each other.
*   Train A is

## Prompt Templates and LCEL

In notebook 20.1 we introduced `ChatPromptTemplate` and LCEL chains. Prompt engineering
is where these tools pay off: a well-crafted template becomes a **reusable, composable**
building block that can be chained with any model or output parser.

The combination of a strong system prompt + a dynamic template + LCEL is the standard
pattern for production LLM components.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# A reusable "expert explainer" template
# The system prompt defines the persona and constraints.
# The user template specifies what varies per call.
explainer_template = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert at explaining complex topics to a {audience}. "
     "Use analogies where helpful. Keep your response under 80 words."),
    ("user", "Explain: {topic}"),
])

explainer_chain = explainer_template | model | StrOutputParser()

examples = [
    {"audience": "10-year-old child",           "topic": "neural networks"},
    {"audience": "experienced software engineer","topic": "neural networks"},
    {"audience": "retired teacher",             "topic": "neural networks"},
]

for ex in examples:
    print(f"=== Audience: {ex['audience']} ===")
    print(explainer_chain.invoke(ex))
    print()

=== Audience: 10-year-old child ===
Think of a neural network like a team of detectives solving a puzzle. Each detective looks for one tiny clue, like a line or a color. They pass their findings to the next group, who combine them to spot shapes. Finally, the last group puts it all together to say, "That’s a cat!" By practicing millions of times, the network learns to recognize patterns, just like your brain learns to recognize your friends’ faces.

=== Audience: experienced software engineer ===
Think of a neural network as a **massive, programmable regression pipeline**. Instead of hard-coding logic, you feed data through layers of weighted connections (neurons). During training, the network uses backpropagation—essentially a distributed gradient descent—to iteratively adjust these weights to minimize error. 

It’s like tuning a complex equalizer: each layer extracts increasingly abstract features, transforming raw input into meaningful patterns. You aren't writing rules; you're opti

## Structured Outputs

Free-text responses are hard to parse reliably. For production use — where downstream code
needs to read specific fields — you want the model to return **typed, validated data**.

LangChain's `.with_structured_output()` accepts a **Pydantic model** as a schema and
guarantees the response conforms to it. No regex parsing, no JSON fences to strip,
no missing fields.

This works by combining the system prompt with instructions about the output schema.
The system prompt still controls *what* the model should do — the schema controls *how*
it structures the result.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class SentimentResult(BaseModel):
    reasoning: str = Field(description="Brief explanation of the sentiment signals found")
    label: Literal["positive", "negative", "neutral"] = Field(description="The sentiment label")
    confidence: float = Field(description="Confidence score between 0.0 and 1.0", ge=0.0, le=1.0)

# .with_structured_output() wraps the model — same interface, typed output
structured_model = model.with_structured_output(SentimentResult)

texts = [
    "I love this app, it has completely changed how I work!",
    "Absolute garbage. Crashes constantly and support is useless.",
    "The update was released on Friday.",
    "The screen quality is great but the battery drains fast.",
]

for text in texts:
    result = structured_model.invoke(
        [SystemMessage("You are a sentiment analysis assistant."),
         HumanMessage(text)]
    )
    print(f"Text      : {text[:65]}")
    print(f"Label     : {result.label}  (confidence: {result.confidence:.2f})")
    print(f"Reasoning : {result.reasoning}")
    print()

Text      : I love this app, it has completely changed how I work!
Label     : positive  (confidence: 1.00)
Reasoning : The user uses the word 'love' and states the app has 'completely changed' their work in a beneficial way, indicating strong satisfaction.

Text      : Absolute garbage. Crashes constantly and support is useless.
Label     : negative  (confidence: 1.00)
Reasoning : The user uses strong negative descriptors like 'absolute garbage', 'crashes constantly', and 'useless' to describe the product and support experience.

Text      : The update was released on Friday.
Label     : neutral  (confidence: 1.00)
Reasoning : The statement is a factual report about the timing of a software update without any emotional language or subjective evaluation.

Text      : The screen quality is great but the battery drains fast.
Label     : neutral  (confidence: 0.90)
Reasoning : The user expresses both positive sentiment regarding screen quality and negative sentiment regarding battery perf

In [ ]:
# More complex schema: extract structured information from unstructured text
from typing import List, Optional

class ProductReviewAnalysis(BaseModel):
    overall_sentiment: Literal["positive", "negative", "mixed", "neutral"]
    mentioned_aspects: List[str] = Field(description="Product aspects explicitly mentioned (e.g. 'battery', 'price')")
    strengths: List[str] = Field(description="Positive points mentioned")
    weaknesses: List[str] = Field(description="Negative points mentioned")
    would_recommend: Optional[bool] = Field(description="Whether the reviewer would recommend the product, or None if unclear")

review_analyzer = model.with_structured_output(ProductReviewAnalysis)

review = (
    "I've been using this laptop for three months. The display is stunning and the keyboard "
    "feels great. Battery life is acceptable — about 6 hours. The fan gets loud under load, "
    "which is annoying. For the price point it's solid, though I wish the webcam were better. "
    "Overall I'd recommend it if you need a budget workhorse."
)

print("Review:")
print(review)
print()

result = review_analyzer.invoke(
    [SystemMessage("You are a product review analyst. Extract structured information from the review."),
     HumanMessage(review)]
)

print(f"Overall sentiment : {result.overall_sentiment}")
print(f"Would recommend   : {result.would_recommend}")
print(f"Aspects mentioned : {result.mentioned_aspects}")
print(f"Strengths         : {result.strengths}")
print(f"Weaknesses        : {result.weaknesses}")

Review:
I've been using this laptop for three months. The display is stunning and the keyboard feels great. Battery life is acceptable — about 6 hours. The fan gets loud under load, which is annoying. For the price point it's solid, though I wish the webcam were better. Overall I'd recommend it if you need a budget workhorse.

Overall sentiment : positive
Would recommend   : True
Aspects mentioned : ['display', 'keyboard', 'battery life', 'fan', 'price', 'webcam']
Strengths         : ['stunning display', 'great keyboard feel', 'solid price point']
Weaknesses        : ['fan gets loud under load', 'poor webcam quality']


## Prompt Sensitivity

LLMs are **sensitive to prompt wording**. Small changes in phrasing — even without changing
the meaning — can produce noticeably different outputs. This has important implications:

- A prompt that works well today may need adjustment as models are updated
- Testing multiple phrasings and picking the best is a standard practice
- The order, tone, and specificity of instructions all affect output quality

Below we test three different ways of asking for the same thing.

In [ ]:
task = "Summarize the following text in one sentence."
text = (
    "LangChain is an open-source framework that simplifies building applications "
    "powered by large language models. It provides standardized interfaces for "
    "chat models, vector stores, tools, and agents, and supports chaining components "
    "together using the LangChain Expression Language (LCEL)."
)

phrasings = {
    "Direct command": (
        "Summarize the following text in one sentence.",
    ),
    "Role + command": (
        "You are a skilled technical writer. "
        "Summarize the following text in exactly one sentence, capturing the core idea.",
    ),
    "Negative constraint": (
        "Summarize the following text in one sentence. "
        "Do NOT start with 'LangChain is'. Do NOT use the word 'simplifies'.",
    ),
}

for style, (system_content,) in phrasings.items():
    print(f"=== {style} ===")
    response = model.invoke([
        SystemMessage(system_content),
        HumanMessage(text),
    ])
    print(response.content)
    print()

=== Direct command ===
[{'type': 'text', 'text': 'LangChain is an open-source framework that streamlines the development of large language model applications by providing standardized interfaces and tools for chaining components together.', 'extras': {'signature': 'EjQKMgG+Pvb7TUmL3qQw4R3Wc8XXj9Gz0X0pihetwIQZQdYeJfM/AyT0ho5wMdC7EUhpyOB4'}}]

=== Role + command ===
[{'type': 'text', 'text': 'LangChain is an open-source framework that streamlines the development of large language model applications by providing standardized interfaces and a flexible composition language for chaining components.', 'extras': {'signature': 'EjQKMgG+Pvb78cEnTgf2Y5p2nIFabYtinfrhisPQSSvVePnpfDQKRzd/bkE6IGZFP71vJYS9'}}]

=== Negative constraint ===
[{'type': 'text', 'text': 'This open-source framework facilitates the development of large language model applications by offering standardized interfaces and a composable architecture for chaining various components.', 'extras': {'signature': 'EjQKMgG+Pvb79AAuxhVGo1

## Temperature and Prompting

**Temperature** controls how deterministic the model is when sampling its next token.
It interacts directly with your prompting strategy:

| Temperature | Behavior | Best used with |
|---|---|---|
| `0.0` | Deterministic, repeatable | Classification, extraction, structured output |
| `0.3–0.7` | Balanced | Q&A, summarization, few-shot classification |
| `1.0+` | Creative, varied | Brainstorming, creative writing, diverse suggestions |

For evaluation and structured output tasks, **always use `temperature=0`** to get
consistent, reproducible results. For creative tasks, raise it.

In [ ]:
creative_prompt = [
    SystemMessage("You are a creative copywriter."),
    HumanMessage("Write a one-sentence tagline for a coffee brand called 'Dawn Brew'."),
]

print("=== temperature=0.0 (deterministic) ===")
cold = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", google_api_key=google_api_key, temperature=0.0)
# Same prompt 3 times — should produce identical results
for _ in range(3):
    print(" -", cold.invoke(creative_prompt).content)

print("\n=== temperature=1.0 (creative) ===")
hot = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", google_api_key=google_api_key, temperature=1.0)
# Same prompt 3 times — produces varied results
for _ in range(3):
    print(" -", hot.invoke(creative_prompt).content)

=== temperature=0.0 (deterministic) ===
 - [{'type': 'text', 'text': 'Awaken your potential with every pour.', 'extras': {'signature': 'EjQKMgG+Pvb7va+vV6w0CyQyMGTXA9hrZrnC58GzEItZy4zO2cmkzfe+ZGPyp8guioL26O0J'}}]
 - [{'type': 'text', 'text': 'Awaken your potential with every pour.', 'extras': {'signature': 'EjQKMgG+Pvb7xBawqUnQ0OHsx92dzDzpyPozpXG3UP0nyleiKxLJ45e3PniO6jjX3ffq6S5Q'}}]
 - [{'type': 'text', 'text': 'Awaken your potential with every pour.', 'extras': {'signature': 'EjQKMgG+Pvb7kp3wSKQ4uyj937SojMgqXhruE5GAjrgcG0spRSn4r13lnF1GOfDz+c99nSjP'}}]

=== temperature=1.0 (creative) ===
 - [{'type': 'text', 'text': 'Awaken your potential with every drop.', 'extras': {'signature': 'EjQKMgG+Pvb7WvlNb+Nsk3D/oZ9hLjD0hr6EzJU/o25CAAYgdXKDECGUNWhy/00/NYrq8LRP'}}]
 - [{'type': 'text', 'text': 'Wake up to the light in every cup.', 'extras': {'signature': 'EjQKMgG+Pvb7Kkf2O+CF4sJ1Mz7UuB2+0foaqDYtJfE5Hg13nR9jPHSzw9WnCMFdz8kBQ84Y'}}]
 - [{'type': 'text', 'text': 'Wake up to the perfect pour.', 'e

In [ ]:
# Temperature=0 for structured tasks: classification should be stable
classification_prompt = [
    SystemMessage(
        "You are a sentiment classifier. "
        "Respond with ONLY one word: Positive, Negative, or Neutral."
    ),
    HumanMessage("The product is fine, nothing special."),
]

print("Classification at temperature=0 (5 runs — should be identical):")
cold_classifier = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview", google_api_key=google_api_key, temperature=0.0
)
results = [cold_classifier.invoke(classification_prompt).content[0]['text'] for _ in range(5)]
print(results)
print(f"All identical: {len(set(results)) == 1}")

Classification at temperature=0 (5 runs — should be identical):
['Neutral', 'Neutral', 'Neutral', 'Neutral', 'Neutral']
All identical: True


## References

- [Prompt Engineering Guide](https://www.promptingguide.ai/)
- [Wei et al. (2022) — Chain-of-Thought Prompting](https://arxiv.org/abs/2201.11903)
- [Kojima et al. (2022) — Zero-Shot CoT ("Let's think step by step")](https://arxiv.org/abs/2205.11916)
- [LangChain: Chat Models](https://python.langchain.com/docs/concepts/chat_models/)
- [LangChain: Prompt Templates](https://python.langchain.com/docs/concepts/prompt_templates/)
- [LangChain: Structured Output](https://python.langchain.com/docs/concepts/structured_outputs/)
- [OpenAI Best Practices for Prompt Engineering](https://platform.openai.com/docs/guides/prompt-engineering)